# iCloud Shared Album Analyzer
**What this does:** Reads every photo & video from an iCloud shared album → runs Claude Haiku vision on each → transcribes videos with Whisper → outputs a Google Sheet with one row per asset (permanent iCloud link + description + transcript + usable-for tag).

**Nothing downloads to your Mac.** Everything runs Colab ↔ Apple CDN ↔ Anthropic ↔ Google Sheets.

## Prerequisites (one-time)
1. iPhone → Photos → the shared album → tap the **person icon** → **Public Website → ON** → copy the `icloud.com/photos/#...` URL
2. A Claude subscription OAuth token — run `claude setup-token` on your Mac
3. A Google account (for Sheets output)

## Run order
Step 1 → 2 → 3 → 4 (paste your values) → 5 → 6 → 7 → 8

## Step 1 — Install dependencies
Run once per Colab session. Takes ~60 seconds.

In [ ]:
# Step 1 — Install all dependencies
import subprocess, sys

print("Installing system deps...")
subprocess.run(["apt-get","-qq","install","-y","ffmpeg"], check=False, capture_output=True)

print("Installing Python packages...")
subprocess.run([sys.executable,"-m","pip","install","-q",
    "openai-whisper",
    "anthropic",
    "google-api-python-client",
    "google-auth",
    "google-auth-httplib2",
    "google-auth-oauthlib",
    "requests",
    "pillow",
], check=True)

import shutil, anthropic, whisper, requests
from PIL import Image
print(f"ffmpeg      : {shutil.which('ffmpeg')}")
print(f"anthropic   : {anthropic.__version__}")
print(f"pillow      : {Image.__version__}")
print("\nALL DEPENDENCIES OK ✓")

## Step 2 — Claude subscription token
Generate with `claude setup-token` on your Mac. Paste below.

In [ ]:
# Step 2 — Set your Claude Code OAuth token
# Run: claude setup-token   on your Mac, paste the sk-ant-oat01-... token below

import os, anthropic

CLAUDE_OAUTH_TOKEN = 'PASTE_YOUR_sk-ant-oat01_TOKEN_HERE'

assert CLAUDE_OAUTH_TOKEN.startswith('sk-ant-oat'), \
    'Token must start with sk-ant-oat. Run  claude setup-token  on your Mac.'

os.environ['CLAUDE_CODE_OAUTH_TOKEN'] = CLAUDE_OAUTH_TOKEN
_OAT = CLAUDE_OAUTH_TOKEN

# sanity check
_client_test = anthropic.Anthropic(auth_token=_OAT)
resp = _client_test.messages.create(
    model='claude-haiku-4-5', max_tokens=10,
    messages=[{'role':'user','content':'Reply: ready'}])
print('Claude auth :', resp.content[0].text.strip())
print('Token OK ✓')

## Step 3 — Google auth
Colab will pop a browser window. Sign in with the Google account where you want the output Sheet created.

In [ ]:
# Step 3 — Google auth (Drive + Sheets)
from google.colab import auth
from googleapiclient.discovery import build
import time

auth.authenticate_user()

drive_service  = build('drive',  'v3')
sheets_service = build('sheets', 'v4')

# shared retry wrapper — used everywhere
def drive_retry(fn, tries=5):
    for i in range(tries):
        try: return fn()
        except Exception as e:
            if i == tries-1: raise
            time.sleep(2**i)

# quick test
me = drive_retry(lambda: drive_service.about().get(fields='user').execute())
print('Signed in as:', me['user']['emailAddress'])
print('Google auth OK ✓')

## Step 4 — Configuration
Paste your iCloud shared album URL and set options.

In [ ]:
# Step 4 — Configuration
# Paste your iCloud shared album URL (must have Public Website enabled)
ICLOUD_URL = 'PASTE_YOUR_icloud.com/sharedalbum/#Token_HERE'

# Name of the Google Sheet to create (or update if it already exists)
SHEET_NAME = 'Tiffany Content Library — iCloud Assets'

# Models
VISION_MODEL   = 'claude-haiku-4-5'
WHISPER_MODEL  = 'medium'  # 'small' = faster, 'medium' = more accurate

# Smoke-test mode: process only first N assets (set to None for full run)
TEST_LIMIT = 3

# Output folder on Drive for the state cache
CACHE_FOLDER_NAME = 'icloud-analyzer-cache'

assert 'icloud.com/photos' in ICLOUD_URL and '#' in ICLOUD_URL, \
    'URL must look like https://www.icloud.com/photos/#AbCdEfGhIjKl'
print(f'Album URL   : {ICLOUD_URL}')
print(f'Sheet name  : {SHEET_NAME}')
print(f'Test limit  : {TEST_LIMIT}')
print('Config OK ✓')

## Step 5 — Fetch iCloud asset list
Calls Apple's CDN API to list every photo and video in the album. No download — metadata only.

In [ ]:
# Step 5 — Fetch iCloud asset list
import re, requests

def parse_token(url):
    m = re.search(r'#([A-Za-z0-9_-]+)', url)
    if not m: raise ValueError(f'Cannot extract token from: {url}')
    return m.group(1)

ALBUM_TOKEN = parse_token(ICLOUD_URL)
ICLOUD_HEADERS = {
    'Origin':       'https://www.icloud.com',
    'Content-Type': 'application/json',
    'User-Agent':   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
}

def fetch_webstream(token):
    """Fetch asset metadata list, handling Apple's partition redirect (330)."""
    base_url = None
    # Try default partition first
    for partition in ['p06','p16','p26','p36','p46','p56','p66','p76']:
        url = f'https://{partition}-sharedstreams.icloud.com/{token}/sharedstreams/webstream'
        try:
            r = requests.post(url, json={'streamCtag': None},
                              headers=ICLOUD_HEADERS, timeout=15)
            if r.status_code == 200:
                return r.json(), f'https://{partition}-sharedstreams.icloud.com/{token}/sharedstreams'
            if r.status_code == 330:
                # Apple redirects to correct partition
                host = r.json().get('X-Apple-MMe-Host','')
                if host:
                    base_url = f'https://{host}/{token}/sharedstreams'
                    r2 = requests.post(f'{base_url}/webstream',
                                       json={'streamCtag': None},
                                       headers=ICLOUD_HEADERS, timeout=15)
                    if r2.status_code == 200:
                        return r2.json(), base_url
            # Try next partition
        except requests.exceptions.RequestException:
            continue
    raise RuntimeError(
        'Could not connect to iCloud. Check:\n'
        '1. Public Website is ON in Photos → album → share button\n'
        '2. The URL is correct (should contain #token at the end)\n'
        '3. The album has at least one photo/video in it')

def fetch_download_urls(base_url, guids):
    """Get signed CDN download URLs for a batch of asset GUIDs."""
    r = requests.post(f'{base_url}/webasseturls',
                      json={'photoGuids': guids},
                      headers=ICLOUD_HEADERS, timeout=30)
    r.raise_for_status()
    return r.json().get('items', {})

print(f'Fetching asset list for token: {ALBUM_TOKEN[:12]}...')
stream_data, BASE_URL = fetch_webstream(ALBUM_TOKEN)

raw_photos = stream_data.get('photos', [])
print(f'Total assets in album: {len(raw_photos)}')

# Parse into clean records
ASSETS = []
for p in raw_photos:
    guid = p.get('photoGuid','')
    ctype = p.get('mediaAssetType','')  # 'video' or 'photo'
    is_video = 'video' in ctype.lower()

    # pick best derivative for download (largest non-original for photos, original for video)
    derivs = p.get('derivatives', {})
    best_key = None
    if is_video:
        # prefer original video
        for k in sorted(derivs.keys(), key=lambda x: int(x) if x.isdigit() else 0, reverse=True):
            if 'video' in derivs[k].get('mediaType','').lower() or k == 'original':
                best_key = k; break
        if not best_key and derivs: best_key = sorted(derivs.keys())[-1]
    else:
        # pick largest jpeg/heic derivative
        for k in sorted(derivs.keys(), key=lambda x: int(x) if x.isdigit() else 0, reverse=True):
            mt = derivs[k].get('mediaType','')
            if 'jpeg' in mt or 'heic' in mt or 'image' in mt:
                best_key = k; break
        if not best_key and derivs: best_key = list(derivs.keys())[0]

    deriv = derivs.get(best_key, {}) if best_key else {}

    ASSETS.append({
        'guid':       guid,
        'is_video':   is_video,
        'date':       p.get('dateCreated',''),
        'filename':   p.get('filename', f'{guid}.{"mov" if is_video else "jpg"}'),
        'width':      deriv.get('width', 0),
        'height':     deriv.get('height', 0),
        'size':       deriv.get('fileSize', 0),
        'deriv_key':  best_key,
        'checksum':   deriv.get('checksum',''),
        # permanent web link (doesn't expire)
        'icloud_link': f'https://www.icloud.com/photos/#{ALBUM_TOKEN}',
    })

photos = sum(1 for a in ASSETS if not a['is_video'])
videos = sum(1 for a in ASSETS if a['is_video'])
print(f'Photos : {photos}')
print(f'Videos : {videos}')

if TEST_LIMIT:
    ASSETS_TO_PROCESS = ASSETS[:TEST_LIMIT]
    print(f'\nSmoke-test mode: processing first {TEST_LIMIT} assets')
    print('Change TEST_LIMIT = None in Step 4 for full run')
else:
    ASSETS_TO_PROCESS = ASSETS
print(f'\nAssets to process: {len(ASSETS_TO_PROCESS)}')
print('Asset list OK ✓')

## Step 6 — Vision + transcript processing
For each asset: fetches from Apple CDN → runs vision → transcribes audio (videos only) → caches result. Resumable — already-processed assets are skipped on re-run.

In [ ]:
# Step 6 — Process every asset (vision + Whisper)
import anthropic, base64, io, json, os, shutil, subprocess, tempfile
from pathlib import Path
from PIL import Image
import requests

_OAT = os.environ.get('CLAUDE_CODE_OAUTH_TOKEN','')
assert _OAT.startswith('sk-ant-oat'), 'Run Step 2 first to set the OAuth token.'
anthro = anthropic.Anthropic(auth_token=_OAT)

# ---- cache on Drive ----
from googleapiclient.http import MediaInMemoryUpload, MediaIoBaseDownload

def find_or_create_folder(name, parent=None):
    q = (f"name='{name}' and mimeType='application/vnd.google-apps.folder' "
         f"and trashed=false")
    if parent: q += f" and '{parent}' in parents"
    r = drive_retry(lambda: drive_service.files().list(
        q=q, fields='files(id)').execute()).get('files',[])
    if r: return r[0]['id']
    body = {'name':name,'mimeType':'application/vnd.google-apps.folder'}
    if parent: body['parents'] = [parent]
    return drive_retry(lambda: drive_service.files().create(
        body=body, fields='id').execute())['id']

CACHE_FOLDER = find_or_create_folder(CACHE_FOLDER_NAME)
print(f'Cache folder: {CACHE_FOLDER}')

def load_cache(guid):
    r = drive_retry(lambda: drive_service.files().list(
        q=f"'{CACHE_FOLDER}' in parents and name='{guid}.json' and trashed=false",
        fields='files(id)').execute()).get('files',[])
    if not r: return None
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive_service.files().get_media(fileId=r[0]['id']))
    done=False
    while not done: _,done = dl.next_chunk()
    return json.loads(buf.getvalue().decode())

def save_cache(guid, data):
    r = drive_retry(lambda: drive_service.files().list(
        q=f"'{CACHE_FOLDER}' in parents and name='{guid}.json' and trashed=false",
        fields='files(id)').execute()).get('files',[])
    media = MediaInMemoryUpload(json.dumps(data,indent=2).encode(), mimetype='application/json')
    if r:
        drive_retry(lambda: drive_service.files().update(
            fileId=r[0]['id'], media_body=media).execute())
    else:
        drive_retry(lambda: drive_service.files().create(
            body={'name':f'{guid}.json','parents':[CACHE_FOLDER]},
            media_body=media, fields='id').execute())

# ---- vision prompts ----
PHOTO_PROMPT = """You are categorizing a photo for Tiffany Parra's YouTube/social content library.
Channel: e-commerce, AI tools, business automation, iOS app "Picsort" (AI camera roll organizer).
Creator: Tiffany, 29, living in Bali. Common subjects: laptop work, phone/app demos, cafes,
beach, scooter, sauna, paddle game, street scenes, product shots, travel, behind-the-scenes.

Return STRICT JSON only:
{
  "topic":       "<picsort-demo|picsort-lifestyle|laptop-work|phone-demo|product|street-scene|food-cafe|nature-travel|paddle|behind-the-scenes|portrait|other>",
  "shot_type":   "<b-roll|portrait|lifestyle|product-shot|screen-recording>",
  "subject":     "<3-8 word noun phrase of main subject>",
  "description": "<one sentence max 25 words: what is in this photo>",
  "usable_for":  "<content idea: thumbnail|b-roll cutaway|instagram post|story background|other>",
  "confidence":  <0.0-1.0>
}"""

VIDEO_PROMPT = """You are categorizing a video clip for Tiffany Parra's YouTube/social library.
Channel: e-commerce, AI tools, business automation, iOS app "Picsort".
Creator: Tiffany, 29, Bali. Day-N building Picsort to $10K/mo series.

{transcript_block}

6-frame grid shown (top-left=start, bottom-right=end).

Return STRICT JSON only:
{
  "topic":       "<picsort-demo|picsort-intro|picsort-hook|picsort-cta|interview|lifestyle|paddle|behind-the-scenes|dialogue|e-commerce|ai-tools|general>",
  "shot_type":   "<a-roll|b-roll|talking-head>",
  "subject":     "<3-8 word noun phrase>",
  "description": "<one sentence max 25 words: what is happening>",
  "usable_for":  "<a-roll main content|b-roll cutaway|hook|cta|voiceover>",
  "confidence":  <0.0-1.0>
}"""

# ---- helpers ----
def fetch_asset_url(guid, deriv_key):
    """Get signed CDN download URL for one asset."""
    data = fetch_download_urls(BASE_URL, [guid])
    item = data.get(guid,{})
    loc  = item.get(deriv_key, item.get(list(item.keys())[0], {})) if item else {}
    url_info = loc.get('url', loc) if isinstance(loc, dict) else {}
    # url_info may be the url string directly or a dict with 'url' key
    if isinstance(url_info, str): return url_info
    return url_info.get('url','')

def fetch_image_b64(url, max_w=1280):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert('RGB')
    if img.width > max_w:
        h = int(img.height * max_w / img.width)
        img = img.resize((max_w, h), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, 'JPEG', quality=82)
    return base64.b64encode(buf.getvalue()).decode()

def make_video_grid(video_url, out_jpg, frames=6, cols=3, rows=2, tw=480, th=270):
    """Stream video from URL, extract N frames, stitch grid."""
    with tempfile.TemporaryDirectory() as td:
        # get duration without downloading full file
        result = subprocess.run([
            'ffprobe','-v','error','-show_entries','format=duration',
            '-of','default=noprint_wrappers=1:nokey=1', video_url],
            capture_output=True, text=True, timeout=30)
        dur = float(result.stdout.strip() or 0)
        if dur <= 0: return False

        starts = [dur*(0.05+0.9*i/(frames-1)) for i in range(frames)]
        tiles = []
        for i,t in enumerate(starts):
            fp = f'{td}/f{i:02d}.jpg'
            subprocess.run([
                'ffmpeg','-y','-ss',f'{t:.2f}','-i',video_url,
                '-frames:v','1','-vf',f'scale={tw}:{th}',
                '-q:v','4', fp],
                check=True, capture_output=True, timeout=30)
            tiles.append(Image.open(fp).convert('RGB'))

        grid = Image.new('RGB',(tw*cols, th*rows))
        for i,im in enumerate(tiles):
            r2,c = divmod(i,cols); grid.paste(im,(c*tw,r2*th))
        grid.save(out_jpg,'JPEG',quality=80)
    return True

def transcribe_video(video_url):
    """Extract audio from video URL, run Whisper. Returns transcript string."""
    with tempfile.TemporaryDirectory() as td:
        audio = f'{td}/audio.mp3'
        result = subprocess.run([
            'ffmpeg','-y','-i',video_url,
            '-vn','-ar','16000','-ac','1','-b:a','64k', audio],
            capture_output=True, timeout=120)
        if result.returncode != 0 or not Path(audio).exists():
            return '[no audio extracted]'
        model = whisper.load_model(WHISPER_MODEL)
        out = model.transcribe(audio, language='en', fp16=False)
        return out.get('text','').strip() or '[no speech]'

def call_vision_photo(b64):
    m = anthro.messages.create(model=VISION_MODEL, max_tokens=400,
        messages=[{'role':'user','content':[
            {'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':b64}},
            {'type':'text','text':PHOTO_PROMPT}]}])
    t = m.content[0].text.strip()
    t = __import__('re').sub(r'^```(?:json)?|```$','',t,flags=8).strip()
    return json.loads(t)

def call_vision_video(grid_jpg, transcript=''):
    tx_block = (f'TRANSCRIPT:\n"{transcript[:500]}"'
                if transcript and transcript not in ['[no speech]','[no audio extracted]']
                else 'NO TRANSCRIPT (silent or no audio)')
    b64 = base64.b64encode(Path(grid_jpg).read_bytes()).decode()
    m = anthro.messages.create(model=VISION_MODEL, max_tokens=400,
        messages=[{'role':'user','content':[
            {'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':b64}},
            {'type':'text','text':VIDEO_PROMPT.format(transcript_block=tx_block)}]}])
    t = m.content[0].text.strip()
    t = __import__('re').sub(r'^```(?:json)?|```$','',t,flags=8).strip()
    return json.loads(t)

# =====================================================================
# MAIN PROCESSING LOOP
# =====================================================================
RESULTS = []   # final rows
work    = Path(tempfile.mkdtemp())

print(f'Processing {len(ASSETS_TO_PROCESS)} assets...\n')

for i, asset in enumerate(ASSETS_TO_PROCESS, 1):
    guid  = asset['guid']
    fname = asset['filename']
    atype = 'video' if asset['is_video'] else 'photo'
    print(f'[{i}/{len(ASSETS_TO_PROCESS)}] {fname} ({atype})', flush=True)

    # check cache
    cached = load_cache(guid)
    if cached and 'error' not in cached:
        print(f'  -> cached: {cached.get("topic","")} | {cached.get("subject","")}')
        RESULTS.append(cached)
        continue

    rec = {**asset, 'type': atype}
    try:
        # get signed download URL
        cdn_url = fetch_asset_url(guid, asset.get('deriv_key',''))
        if not cdn_url:
            raise RuntimeError('Could not get CDN download URL')
        rec['cdn_url'] = cdn_url

        if not asset['is_video']:
            # PHOTO: fetch + vision
            b64 = fetch_image_b64(cdn_url)
            v   = call_vision_photo(b64)
            rec.update(v)
            rec['transcript'] = '—'
            print(f'  -> {v.get("topic","")} | {v.get("subject","")} (conf={v.get("confidence",0):.2f})')
        else:
            # VIDEO: grid vision + Whisper transcript
            grid = work/f'{guid}.jpg'
            has_grid = make_video_grid(cdn_url, str(grid))

            print(f'  transcribing...', end=' ', flush=True)
            transcript = transcribe_video(cdn_url)
            print(f'done ({len(transcript)} chars)')
            rec['transcript'] = transcript

            if has_grid:
                v = call_vision_video(str(grid), transcript)
                rec.update(v)
            else:
                rec.update({'topic':'unknown','shot_type':'unknown',
                            'subject':'zero-duration clip','description':'',
                            'usable_for':'review','confidence':0.0})
            print(f'  -> {rec.get("topic","")} / {rec.get("shot_type","")} | {rec.get("subject","")}')
            if grid.exists(): grid.unlink()

        save_cache(guid, rec)
        RESULTS.append(rec)

    except Exception as e:
        print(f'  ERROR: {e}')
        rec['error'] = str(e)[:300]
        save_cache(guid, rec)
        RESULTS.append(rec)

shutil.rmtree(work, ignore_errors=True)
ok  = sum(1 for r in RESULTS if 'error' not in r)
err = sum(1 for r in RESULTS if 'error' in r)
print(f'\nDone. OK={ok}  Errors={err}')
print('Processing complete ✓')

## Step 7 — Write to Google Sheet
Creates or updates a Google Sheet with one row per asset. Editors can click the iCloud link on any row to view the original photo/video directly.

In [ ]:
# Step 7 — Write results to Google Sheet
import os

_OAT = os.environ.get('CLAUDE_CODE_OAUTH_TOKEN','')
assert _OAT.startswith('sk-ant-oat'), 'Run Step 2 first.'

# ---- find or create the Sheet ----
def find_sheet(name):
    r = drive_retry(lambda: drive_service.files().list(
        q=f"name='{name}' and mimeType='application/vnd.google-apps.spreadsheet' "          f"and trashed=false",
        fields='files(id,webViewLink)').execute()).get('files',[])
    return (r[0]['id'], r[0]['webViewLink']) if r else (None, None)

def create_sheet(name):
    r = drive_retry(lambda: drive_service.files().create(
        body={'name':name,'mimeType':'application/vnd.google-apps.spreadsheet'},
        fields='id,webViewLink').execute())
    return r['id'], r['webViewLink']

sheet_id, sheet_url = find_sheet(SHEET_NAME)
if not sheet_id:
    sheet_id, sheet_url = create_sheet(SHEET_NAME)
    print(f'Created sheet: {sheet_url}')
else:
    print(f'Updating existing sheet: {sheet_url}')

# ---- build rows ----
HEADERS = [
    '#', 'clip_id', 'filename', 'type', 'date',
    'topic', 'shot_type', 'subject', 'description',
    'transcript', 'usable_for', 'confidence',
    'icloud_link', 'width', 'height', 'size_bytes'
]

def fmt_date(d):
    # iCloud dates: '2024-05-01T10:23:45Z' or similar
    return d[:10] if d else ''

def fmt_link(url, label='View ↗'):
    if not url: return ''
    return f'=HYPERLINK("{url}","{label}")'

sheet_rows = [HEADERS]
for idx, r in enumerate(RESULTS, 1):
    sheet_rows.append([
        idx,
        r.get('guid','')[:8],
        r.get('filename',''),
        r.get('type',''),
        fmt_date(r.get('date','')),
        r.get('topic','') if 'error' not in r else f'ERROR: {r["error"][:60]}',
        r.get('shot_type',''),
        r.get('subject',''),
        r.get('description',''),
        (r.get('transcript','') or '')[:500],  # cap at 500 chars in cell
        r.get('usable_for',''),
        str(round(r.get('confidence',0), 2)) if 'confidence' in r else '',
        fmt_link(r.get('icloud_link','')),
        r.get('width',''),
        r.get('height',''),
        r.get('size',''),
    ])

# ---- write to Sheet ----
RANGE = f'Sheet1!A1:{chr(65+len(HEADERS)-1)}{len(sheet_rows)}'

# clear existing content first
drive_retry(lambda: sheets_service.spreadsheets().values().clear(
    spreadsheetId=sheet_id, range='Sheet1').execute())

drive_retry(lambda: sheets_service.spreadsheets().values().update(
    spreadsheetId=sheet_id,
    range=RANGE,
    valueInputOption='USER_ENTERED',
    body={'values': sheet_rows}).execute())

# ---- format: bold header, freeze row 1, auto-resize ----
reqs = [
    # bold header
    {'repeatCell':{
        'range':{'sheetId':0,'startRowIndex':0,'endRowIndex':1},
        'cell':{'userEnteredFormat':{'textFormat':{'bold':True},
                                    'backgroundColor':{'red':0.2,'green':0.2,'blue':0.2},
                                    'foregroundColor':{'red':1,'green':1,'blue':1}}},
        'fields':'userEnteredFormat(textFormat,backgroundColor,foregroundColor)'}},
    # freeze row 1
    {'updateSheetProperties':{
        'properties':{'sheetId':0,'gridProperties':{'frozenRowCount':1}},
        'fields':'gridProperties.frozenRowCount'}},
    # auto-resize all columns
    {'autoResizeDimensions':{
        'dimensions':{'sheetId':0,'dimension':'COLUMNS',
                      'startIndex':0,'endIndex':len(HEADERS)}}},
]
drive_retry(lambda: sheets_service.spreadsheets().batchUpdate(
    spreadsheetId=sheet_id,
    body={'requests':reqs}).execute())

print(f'\n✅ Sheet updated: {len(sheet_rows)-1} asset rows written')
print(f'\n🔗 Open your sheet here:')
print(f'   {sheet_url}')
SHEET_URL = sheet_url

## Step 8 — Summary + CSV download
Shows a breakdown of what was found and downloads a local CSV backup.

In [ ]:
# Step 8 — Summary + CSV download
import csv, io, os
from collections import Counter

print('='*60)
print(f'ICLOUD ALBUM ANALYSIS COMPLETE')
print('='*60)
print(f'Total assets processed : {len(RESULTS)}')
print(f'Photos                 : {sum(1 for r in RESULTS if r.get("type")=="photo")}')
print(f'Videos                 : {sum(1 for r in RESULTS if r.get("type")=="video")}')
print(f'Errors                 : {sum(1 for r in RESULTS if "error" in r)}')

ok = [r for r in RESULTS if 'error' not in r]
if ok:
    print()
    print('--- by topic ---')
    for t,n in Counter(r.get('topic','') for r in ok).most_common():
        print(f'  {t:<28} {n}')
    print()
    print('--- by usable_for ---')
    for t,n in Counter(r.get('usable_for','') for r in ok).most_common():
        print(f'  {t:<28} {n}')

print()
print(f'📊 Google Sheet : {SHEET_URL}')
print()

# CSV backup
fields = ['guid','filename','type','date','topic','shot_type',
          'subject','description','transcript','usable_for',
          'confidence','icloud_link','width','height','size']
local_csv = '/content/icloud-content-catalog.csv'
with open(local_csv,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader()
    for r in RESULTS:
        row = {k: r.get(k,'') for k in fields}
        row['transcript'] = (row.get('transcript','') or '')[:1000]
        w.writerow(row)
print(f'CSV saved: {local_csv}')

try:
    from google.colab import files
    files.download(local_csv)
except Exception:
    print('(Run in Colab to auto-download)')

print()
print('NEXT STEP: Paste the CSV into Claude Code with:')
print('  "Based on this content library, plan a 30-day content calendar')
print('   for Tiffany\'s YouTube channel (e-commerce, AI tools, Picsort app)."')